# Inativar Títulos Vencidos

Este notebook atualiza o status de todos os títulos públicos com `data_vencimento` anterior à data atual, marcando-os como `INATIVO`.

In [ ]:
from datetime import date
from dotenv import load_dotenv

from db.connection import get_conn

load_dotenv()

True

In [ ]:
# Data de hoje (formato ISO: YYYY-MM-DD)
hoje = date.today().isoformat()
print(f"Data de referência: {hoje}")

Data de referência: 2026-01-22


In [ ]:
# Verificar quantos títulos serão inativados
with get_conn() as conn:
    cursor = conn.execute("""
        SELECT COUNT(*) 
        FROM TITULOS_PUBLICOS 
        WHERE data_vencimento < ? 
        AND status = 'ATIVO'
    """, (hoje,))
    count = cursor.fetchone()[0]
    print(f"Títulos ativos com vencimento < {hoje}: {count}")

Títulos ativos com vencimento < 2026-01-22: 66


In [ ]:
# Ver alguns exemplos antes de atualizar
with get_conn() as conn:
    cursor = conn.execute("""
        SELECT id, tipo_titulo, data_vencimento, status
        FROM TITULOS_PUBLICOS 
        WHERE data_vencimento < ? 
        AND status = 'ATIVO'
        ORDER BY data_vencimento DESC
        LIMIT 10
    """, (hoje,))
    rows = cursor.fetchall()
    print("Exemplos de títulos que serão inativados:")
    print("-" * 60)
    for row in rows:
        print(f"ID: {row[0]:4d} | {row[1]:10s} | {row[2]} | {row[3]}")
    if len(rows) == 0:
        print("Nenhum título encontrado.")

Exemplos de títulos que serão inativados:
------------------------------------------------------------
ID:   86 | LTN        | 2026-01-01 | ATIVO
ID:   92 | LTN        | 2025-10-01 | ATIVO
ID:   60 | LFT        | 2025-09-01 | ATIVO
ID:   79 | LTN        | 2025-07-01 | ATIVO
ID:   66 | NTN-B      | 2025-05-15 | ATIVO
ID:   89 | LTN        | 2025-04-01 | ATIVO
ID:   58 | LFT        | 2025-03-01 | ATIVO
ID:   37 | NTN-F      | 2025-01-01 | ATIVO
ID:   73 | LTN        | 2025-01-01 | ATIVO
ID:   85 | LTN        | 2024-10-01 | ATIVO


In [ ]:
# Atualizar status para INATIVO
with get_conn() as conn:
    cursor = conn.execute("""
        UPDATE TITULOS_PUBLICOS 
        SET status = 'INATIVO'
        WHERE data_vencimento < ? 
        AND status = 'ATIVO'
    """, (hoje,))
    conn.commit()
    atualizados = cursor.rowcount
    print(f"✅ {atualizados} títulos atualizados para INATIVO")

✅ 66 títulos atualizados para INATIVO


In [ ]:
# Verificar resultado final
with get_conn() as conn:
    cursor = conn.execute("""
        SELECT status, COUNT(*) 
        FROM TITULOS_PUBLICOS 
        GROUP BY status
        ORDER BY status
    """)
    print("\nDistribuição por status:")
    print("-" * 30)
    for row in cursor.fetchall():
        print(f"{row[0]:12s}: {row[1]:6d}")


Distribuição por status:
------------------------------
ATIVO       :     52
INATIVO     :     66
